# 4 - Guide d'intégration des modèles de différents packages

## Importation des modules

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Importation des modules
# Modules de base
import numpy as np
import pandas as pd
import sys

# Ajout du chemin
sys.path.append('..')

# Importation des utilitaires sklearn
from sklearn.utils import _safe_indexing
from sklearn.utils.metaestimators import _safe_split
from sklearn.model_selection import cross_val_predict

# Importation des modèles
# Sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
# XGBoost

# Importation des pipelines
# Sklearn
from sklearn.pipeline import Pipeline

#  Eléments du package à intégrer
# Crossval
from tsforecast.crossvals import (
    PanelOutOfSampleSplit
)

## 1. Création de données synthétiques

### 1.1 Génération de données de panel

In [ ]:
# Fonction de génération des données de panel
def generate_panel_data(entities=['A', 'B', 'C'], start_date='2015-01-01', periods=120, 
                        freq='MS', heterogeneous_effects=True, common_trend=True, 
                        entity_specific_seasonality=True, cross_sectional_correlation=0.3,
                        missing_data_prob=0.0):
    """
    Generate synthetic panel data with various characteristics.
    
    Args:
        entities: List of entity identifiers
        start_date: Start date for the panel
        periods: Number of time periods per entity
        freq: Frequency of observations
        heterogeneous_effects: Whether entities have different baseline levels
        common_trend: Whether to include a common trend across entities
        entity_specific_seasonality: Whether seasonality patterns differ by entity
        cross_sectional_correlation: Correlation between entity shocks
        missing_data_prob: Probability of missing observations
    
    Returns:
        pd.DataFrame: Panel data with MultiIndex (entity, date)
    """
    # Création de l'index temporel
    dates = pd.date_range(start=start_date, periods=periods, freq=freq)
    
    # Création du MultiIndex (entity, date)
    index = pd.MultiIndex.from_product([entities, dates], names=['entity', 'date'])
    
    # Initialisation du DataFrame
    panel_data = pd.DataFrame(index=index)
    
    # Génération des effets fixes par entité (hétérogénéité)
    if heterogeneous_effects:
        entity_effects = {entity: np.random.normal(0, 2) for entity in entities}
    else:
        entity_effects = {entity: 0 for entity in entities}
    
    # Tendance commune
    if common_trend:
        common_trend_values = 0.02 * np.arange(periods)
    else:
        common_trend_values = np.zeros(periods)
    
    # Génération de chocs corrélés entre entités
    if cross_sectional_correlation > 0:
        # Chocs communs
        common_shocks = np.random.normal(0, 1, periods)
        # Chocs idiosyncratiques
        idiosyncratic_shocks = {
            entity: np.random.normal(0, 1, periods) 
            for entity in entities
        }
    
    # Construction des séries pour chaque entité
    values = []
    
    # Parcours des entités
    for entity in entities:
        # Effet fixe de l'entité
        entity_effect = entity_effects[entity]
        
        # Saisonnalité spécifique à l'entité
        if entity_specific_seasonality:
            # Période et amplitude différentes selon l'entité
            seasonal_period = 20 + hash(entity) % 40  # Entre 20 et 60
            seasonal_amplitude = 0.5 + (hash(entity) % 100) / 200  # Entre 0.5 et 1.0
        else:
            seasonal_period = 30
            seasonal_amplitude = 0.5
        
        seasonal_values = seasonal_amplitude * np.sin(2 * np.pi * np.arange(periods) / seasonal_period)
        
        # Processus autorégressif spécifique à l'entité
        ar_coef = 0.5 + (hash(entity) % 50) / 100  # Entre 0.5 et 1.0
        ar_process = np.zeros(periods)
        ar_process[0] = np.random.normal(0, 0.5)
        for t in range(1, periods):
            ar_process[t] = ar_coef * ar_process[t-1] + np.random.normal(0, 0.5)
        
        # Combinaison des composantes
        if cross_sectional_correlation > 0:
            # Chocs avec corrélation croisée
            correlated_shocks = (
                np.sqrt(cross_sectional_correlation) * common_shocks +
                np.sqrt(1 - cross_sectional_correlation) * idiosyncratic_shocks[entity]
            )
        else:
            correlated_shocks = np.random.normal(0, 1, periods)
        
        # Combinaison des valeurs
        entity_values = (
            entity_effect + 
            common_trend_values + 
            seasonal_values + 
            ar_process + 
            correlated_shocks
        )
        
        # Ajout de données manquantes
        if missing_data_prob > 0:
            missing_mask = np.random.random(periods) < missing_data_prob
            entity_values[missing_mask] = np.nan
        
        values.extend(entity_values)
    
    # Création du DataFrame final avec les valeurs
    panel_data['value'] = values
    
    # Ajout de variables explicatives
    panel_data['lag_value'] = panel_data.groupby('entity')['value'].shift(1)
    panel_data['trend'] = np.tile(np.arange(periods), len(entities))
    panel_data['month'] = panel_data.index.get_level_values('date').month
    
    return panel_data

In [ ]:
# Génération de différents types de données de panel
print("📊 Génération de données de panel ...")

# Panel 1: Données équilibrées avec effets hétérogènes
entities_small = ['FR', 'DE', 'IT', 'ES']
df_panel = generate_panel_data(
    entities=entities_small,
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
    common_trend=True,
    entity_specific_seasonality=True,
    cross_sectional_correlation=0.4
)

# Suppression des Nan
df_panel.dropna(how='any', inplace=True)

print(f"✅ Génération de panels terminée:")
print(f"Caractéristiques des données générées : {df_panel.shape[0]} observations, {len(entities_small)} entités")

# Affichage des premières observations de chaque panel
print(f"\n📋 Aperçu des données:")
print(df_panel.head(10))

### 1.2 Génération de données hiérarchiques

Pour illustrer la réconciliation hiérarchique, on génère des séries `value_b` et `value_c` 
telles que `value_a = value_b + value_c`. Cette contrainte d'agrégation est la base 
des hiérarchies cross-sectionnelles : les prévisions indépendantes de chaque composante 
ne respectent pas nécessairement la contrainte, et la réconciliation permet de les rendre cohérentes.

In [ ]:
def generate_hierarchical_data(
    entities=['X', 'Y', 'Z'],
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
    seed=42,
):
    """Generate hierarchical panel data where value_a = value_b + value_c.

    Generates two bottom-level series (value_b, value_c) per entity, with
    distinct trend, seasonality and noise patterns. The top-level series
    (value_a) is their exact sum, forming a cross-sectional aggregation
    constraint. Data is indexed by (entity, date).

    Args:
        entities: List of entity identifiers.
        start_date: Start date for the time series.
        periods: Number of time periods per entity.
        freq: Frequency of observations.
        heterogeneous_effects: Whether entities have different baseline levels
            and trend slopes for value_b and value_c.
        seed: Random seed for reproducibility.

    Returns:
        Tuple of (df_wide, df_long) where:
            - df_wide: DataFrame with MultiIndex (entity, date) and columns
              value_a, value_b, value_c.
            - df_long: DataFrame with MultiIndex (entity, category, component, date)
              and column y, formatted for hierarchicalforecast cross-sectional
              aggregation.
    """
    np.random.seed(seed)
    dates = pd.date_range(start=start_date, periods=periods, freq=freq)
    t = np.arange(periods)

    wide_records = []
    long_records = []

    for entity in entities:
        # Effets fixes par entité (niveaux et pentes)
        if heterogeneous_effects:
            base_b = np.random.uniform(40, 70)
            base_c = np.random.uniform(20, 45)
            slope_b = np.random.uniform(0.05, 0.25)
            slope_c = np.random.uniform(-0.10, 0.05)
        else:
            base_b, base_c = 50, 30
            slope_b, slope_c = 0.15, -0.05

        # Composante B : tendance + saisonnalité annuelle + bruit
        value_b = (
            base_b
            + slope_b * t
            + 5 * np.sin(2 * np.pi * t / 12)
            + np.random.normal(0, 1.5, periods)
        )

        # Composante C : tendance + saisonnalité semestrielle + bruit
        value_c = (
            base_c
            + slope_c * t
            + 3 * np.sin(2 * np.pi * t / 6)
            + np.random.normal(0, 1.0, periods)
        )

        # Agrégat : contrainte d'agrégation exacte
        value_a = value_b + value_c

        # Accumulation des lignes pour le format wide
        for d, va, vb, vc in zip(dates, value_a, value_b, value_c):
            wide_records.append({
                'entity': entity, 'date': d,
                'value_a': va, 'value_b': vb, 'value_c': vc,
            })

        # Accumulation des lignes pour le format long
        for d, vb, vc in zip(dates, value_b, value_c):
            long_records.append(
                {'entity': entity, 'category': 'Total', 'component': 'B', 'date': d, 'y': vb}
            )
            long_records.append(
                {'entity': entity, 'category': 'Total', 'component': 'C', 'date': d, 'y': vc}
            )

    # Format wide : MultiIndex (entity, date)
    df_wide = pd.DataFrame(wide_records).set_index(['entity', 'date'])

    # Format long : MultiIndex (entity, category, component, date)
    df_long = pd.DataFrame(long_records).set_index(['entity', 'category', 'component', 'date'])

    return df_wide, df_long


# Génération des données hiérarchiques
print('📊 Génération de données hiérarchiques ...')
df_hier_wide, df_hier_long = generate_hierarchical_data(
    entities=['FR', 'DE', 'IT', 'ES'],
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
)

n_entities = df_hier_wide.index.get_level_values('entity').nunique()
constraint_ok = (
    df_hier_wide
    .assign(check=lambda d: np.isclose(d['value_a'], d['value_b'] + d['value_c']))
    ['check']
    .all()
)

print('✅ Données hiérarchiques générées:')
print(f'   Format wide : {df_hier_wide.shape} (MultiIndex entity/date, {n_entities} entités)')
print(f'   Format long  : {df_hier_long.shape} (MultiIndex entity/category/component/date)')
print(f'   Vérification : value_a == value_b + value_c → {constraint_ok}')

print('\n📋 Aperçu format wide :')
print(df_hier_wide.head(8))
print('\n📋 Aperçu format long :')
print(df_hier_long.head(8))

## 2. Création d'un cadre de prévision à partir des éléments développpés dans `tsforecast`

In [ ]:
# Séparation en X et y
y = df_panel['value'].copy()
X = df_panel.drop('value', axis=1)

# Initialisation de l'horizon de prédiction
horizon=2
# Initialisation du délai de publication
delays=1
# Application de l'horizon aux données afin d'aligner X et y à prévoir
# /!\ Créer une classe plus intelligente qui utilise la régularité de la série (sur données de panel et de séries temporelles) pour ajouter les dates manquantes aux extrémités de la période et ne pas perdre de données
X = X.shift(-horizon)

# Initialisation de la crossval pour l'ensemble des tests
cv = PanelOutOfSampleSplit(
    test_indices=['2024-01-01', '2024-02-01'], 
    test_size=1, 
    gap=horizon + delays
)
splits = list(cv.split(X, y))
# Extraction des indices d'entrainement et de test
train, test = splits[0]

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_indexing(X, train), _safe_indexing(y, train)
X_test = _safe_indexing(X, test)

# Construction d'une seconde crossval pour la validation croisée 
cv_val = PanelOutOfSampleSplit(
    test_indices=['2023-06-01', '2023-07-01', '2023-08-01'],
    test_size=1,
    gap=horizon + delays
)

# Affichage des jeux de données
# X_train
print("="*10)
print("X_train")
print("="*10)
print(X_train.tail())
# y_train
print("="*10)
print("y_train")
print("="*10)
print(y_train.tail())
# X_test
print("="*10)
print("X_test")
print("="*10)
print(X_test.head())

## 3. Intégration des modèles de différents packages dans un cadre unifié, compatible avec les éléments développés dans `tsforecast`

### 3.1. Intégration des modèles `sklearn`

L'intégration de l'ensemble des estimateurs de `sklearn` dans le workflow se fait nativement, les observations `X` et `y` étant déjà alignées. La syntaxe est ainsi celle de `sklearn` :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... peuvent être utilisés de la même manière en respectant la syntaxe originale du package.

In [ ]:
# Importation du modèle
from sklearn.linear_model import LinearRegression

# Initialisation du modèle
estimator=LinearRegression()

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sklearn = estimator.predict(X_test)

y_pred_sklearn

L'intégration des `Pipeline` de `sklearn.pipeline`, permettant de combiner des transformations opérées sur les données et l'entraînement d'un estimateur en fin de processus, se fait de la même manière en respectant la syntaxe originale.

In [ ]:
# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('RidgeEstimator', LinearRegression())
])

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sklearn_pipeline = estimator.predict(X_test)

y_pred_sklearn_pipeline

### 3.2. Intégration des modèles `xgboost`

L'API de `xgboost` est entièrement compatible avec celle de `sklearn`, aussi ses estimateurs s'intègrent nativement dans un workflow similaire. La syntaxe est ainsi celle de `sklearn` :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... de `sklearn` peuvent être utilisés avec ces modèles sans modification de la syntaxe.

In [ ]:
# Importation du modèle
from xgboost import XGBRegressor

# Initialisation du modèle
estimator=XGBRegressor()

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_xgboost = estimator.predict(X_test)

y_pred_xgboost

 La combinaison de transformers avec les estimateurs de `xgboost` dans une `Pipeline` de `sklearn.pipeline` s'opère également sans difficulté.

In [ ]:
# Importation du modèle
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('XGBoostEstimator', XGBRegressor())
])

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_xgboost_pipeline = estimator.predict(X_test)

y_pred_xgboost_pipeline

### 3.3. Intégration des modèles `tslearn`

`tslearn` reprend également l'API de `sklearn`, aussi ses estimateurs s'intègrent nativement dans un workflow similaire. La syntaxe est ainsi celle de `sklearn` :
- `.fit(X,y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... de `sklearn` peuvent être utilisés avec ces modèles sans modification de la syntaxe.

In [ ]:
# Importation du modèle
from tslearn.svm import TimeSeriesSVR

# Initialisation du modèle
estimator=TimeSeriesSVR()

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_tslearn = estimator.predict(X_test)

y_pred_tslearn

La combinaison de ces modèles avec des transformers de `sklearn` dans une `Pipeline` de `sklearn.Pipeline` s'opère de manière transparente

In [ ]:
# Importation du modèle
from sklearn.preprocessing import StandardScaler
from tslearn.svm import TimeSeriesSVR

# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('SVREstimator', TimeSeriesSVR())
])

# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_tslearn_pipeline = estimator.predict(X_test)

y_pred_tslearn_pipeline

### 3.4. Intégration des modèles `sktime`

Les modèles issus de des modules  `sktime.regression`, `sktime.classification` suivent l'API de `sklearn` mais sont rarement pertinents car ils nécessitent le plus souvent la définition d'une `window` qui peut venir dégrader les performances du modèle. Cette intégration n'est pas prévue dans `tsforecast`. Il est souvent préférable d'utiliser l'équivalent dans `sklearn`.

Les transformers de `sktime.transformations` s'intègrent eux sans difficulté dans le workflow avec la syntaxe :
- `.fit(X,y)` pour l'entraînement ;
- `.transform(X)` pour la transform ;

L'ensemble des utilitaires comme `GridSearchCV`, `cross_val_score` etc ... de `sklearn` peuvent être utilisés avec ces transformers sans modification de la syntaxe.

Les forecasters ont une syntaxe qui diffère légèrement de celle des modèles du package `sklearn` en ce que les méthodes `fit` et `predict` ont également pour argument un horizon de prédiction. Si l'horizon de prédiction est spécifié lors de l'entrainement du modèle, il n'a pas besoin de l'être à nouveau pour la prévision. La syntaxe des `forecasters` est ainsi la suivante :
- `.fit(X,y, fh)` pour l'entraînement ;
- `.predict(X, fh)` pour la prédiction ;

#### 3.4.1 Utilisation des modèles de `sktime.forecasting`

L'horizon de prédiction étant déjà appliqué en faisant un `shift`/ `Lag` sur les données pour aligner les covariables `X` avec la valeur de `y` à prévoir, correspondant à un horizon de prédiction donné, il est inutile d'appliquer cette logique de décalage une nouvelle fois. Aussi, l'intégration dans la syntaxe `sklearn` des forecasters de sktime peut se faire en spécifiant `fh=0` comme argument de la méthode `fit` et en utilisant l'adapter `SktimeAdapter` qui permet de convertir les données `X` et `y` au format attendu par les forecasters.

Les utilitaires de `sklearn` peuvent s'intégrer dans cette syntaxe :
- `sklearn.pipeline.Pipeline` possède une méthode `fit(X, y=None, **params)` qui permet la spécification de `fh=0` qui est ensuite transmis comme argument de la méthode `fit` du forecaster. `sktime` a également implémenté sa version de cet utilitaire à travers `sktime.pipeline.Pipeline` qui possède la même signature qu'une pipeline `sklearn` et tolère estimateurs classiques et forecasters comme étapes ainsi que `sktime.forecasting.compose.ForecastingPipeline` qui possède la même syntaxe qu'forecaster classique pour ses méthodes `fit` et `predict`
- `sklearn.model_selection.GridSearchCV` possède également une méthode `fit(X, y=None, **params)` qui permet la spécification de `fh=0`. `sktime` a également implémenté sa version de cet utilitaire à travers `sktime.forecasting.model_selection.ForecastingGridSearchCV` qui possède la même signature qu'un forecaster classique pour ses méthodes `fit` et `predict`
- `sklearn.model_selection.cross_val_score` possède un argument `params` (`sklearn.model_selection.cross_val_predict`possède un argument `fit_params` qui poursuit le même objectif) qui permet de spécifier sous la forme d'un dictionnaire des paramètres de la méthode `fit` de l'estimateur. En l'occurrence `{'fh' : 0}` permet de spécifier le comportement attendu

In [ ]:
# Importation du modèle
from sktime.forecasting.naive import NaiveForecaster
# Importation de l'adapter
from tsforecast.adapters import SktimeAdapter

# Initialisation du modèle
estimator=SktimeAdapter(forecaster=NaiveForecaster())

# Entraînement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_forecaster_sktime = estimator.predict(X_test)

y_pred_forecaster_sktime

In [ ]:
# Importation de cross_val_score
from sklearn.model_selection import cross_val_score
# Importation du modèle
from sktime.forecasting.naive import NaiveForecaster

# Calcul du score
score_forecaster_sktime = cross_val_score(
    estimator=SktimeAdapter(forecaster=NaiveForecaster()),
    X=X,
    y=y,
    cv=cv,
    n_jobs=-1,
)

score_forecaster_sktime

`sklearn.pipeline.Pipeline` peut être utilisé avec des forecasters `sktime` en transmettant `fh=0` via les paramètres de fit.

In [ ]:
# Importation de la pipeline
from sklearn.pipeline import Pipeline
# Importation du transformer
from sklearn.preprocessing import StandardScaler
# Importation du modèle
from sktime.forecasting.naive import NaiveForecaster
# Importation de l'adapter
from tsforecast.adapters import SktimeAdapter

# Construction de la pipeline avec preprocessing sklearn et forecaster sktime
pipeline_sklearn = Pipeline([
    ('scaler', StandardScaler()),  # Preprocessing sklearn
    ('forecaster', SktimeAdapter(forecaster=NaiveForecaster()))  # Forecaster sktime
])

# Entraînement de la pipeline avec fh=0
pipeline_sklearn.fit(X_train, y_train)

# Prédiction
y_pred_pipeline_sklearn_sktime = pipeline_sklearn.predict(X_test)

y_pred_pipeline_sklearn_sktime

`GridSearchCV` de sklearn peut être utilisé pour optimiser les hyperparamètres d'un forecaster `sktime`.

In [ ]:
# Importation de la gridsearch
from sklearn.model_selection import GridSearchCV
# Importation de la pipeline
from sklearn.pipeline import Pipeline
# Importation du transformer
from sklearn.preprocessing import StandardScaler
# Importation du modèle
from sktime.forecasting.naive import NaiveForecaster
# Importation de l'adapter
from tsforecast.adapters import SktimeAdapter

# Définition de la pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('forecaster', SktimeAdapter(forecaster=NaiveForecaster()))
])

# Définition de la grille de paramètres
param_grid = {
    'forecaster__forecaster__strategy': ["last", "mean", "drift"],
}

# Initialisation du GridSearchCV
grid_search_sklearn = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_val,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)

# Entraînement avec fh=0
grid_search_sklearn.fit(X_train, y_train)

# Meilleurs paramètres et score
print(f"Meilleurs paramètres: {grid_search_sklearn.best_params_}")
print(f"Meilleur score: {grid_search_sklearn.best_score_}")

# Prédiction avec le meilleur modèle
y_pred_grid_sklearn = grid_search_sklearn.predict(X_test)

y_pred_grid_sklearn

#### 3.4.2. Utilisation des forecasters comme plateforme d'intégration des modèles de différents packages

`make_reduction` transforme n'importe quel régresseur (issu de `sklearn`, `xgboost` etc ...) en forecaster via des **stratégies de réduction** :
- `recursive`: Un modèle, réutilise prédictions pour multi-step (rapide, peut accumuler erreurs)
- `direct`: Un modèle par horizon (précis sur long terme, plus lent)
- `multioutput`: Un modèle prédit tous horizons simultanément

Le sliding window est automatique: `[y[t-window_length], ..., y[t-1]] → y[t]`. 

**Cependant `make_reduction` n'est pas compatible avec un horizon de prédiction nul (fh=0)**.

Il est cependant possible d'utiliser `YfromX` qui crée des forecasters où `X` prédit `y` directement **sans lags autorégressifs**.

**Différence clé:** `make_reduction` utilise des lags de `y` comme features. `YfromX` utilise `X` uniquement sans historique de `y`.

In [ ]:
# Importation de XfromX
from sktime.forecasting.compose import YfromX
# Importation du modèle
from sklearn.ensemble import RandomForestRegressor
# Importation de l'adapter
from tsforecast.adapters import SktimeAdapter

# Création d'un forecaster qui utilise uniquement X (sans lags de y)
forecaster_rf_yfromx = SktimeAdapter(
    forecaster=YfromX(
        estimator=RandomForestRegressor(n_estimators=100, random_state=42)
    )
)

# Entraînement du modèle
# Ici, le modèle apprend directement la relation X -> y
forecaster_rf_yfromx.fit(X_train, y_train)

# Prédiction
y_pred_rf_yfromx = forecaster_rf_yfromx.predict(X_test)

y_pred_rf_yfromx

### 3.5. Intégration des modèles `darts`

L'API de `darts` diffère de celle de `sklearn` en ce que les modèles manipulent des objets `TimeSeries` propres au package plutôt que des DataFrames pandas. Pour intégrer ces modèles dans un workflow unifié, le package `tsforecast` fournit un adapter `DartsAdapter` qui encapsule les modèles `darts` dans une interface compatible `sklearn`.

La syntaxe devient alors celle de `sklearn` :
- `.fit(X, y)` pour l'entraînement ;
- `.predict(X)` pour la prédiction ;
- `.score(X, y)` pour l'évaluation ;

L'adapter gère automatiquement la conversion entre pandas et TimeSeries de darts. L'ensemble des utilitaires `sklearn` comme `GridSearchCV`, `cross_val_score`, `Pipeline` etc. peuvent être utilisés de manière transparente.

In [ ]:
# Importation de l'adapter
from tsforecast.adapters import DartsAdapter
# Importation d'un modèle darts
from darts.models import LinearRegressionModel

# Initialisation du modèle darts
darts_model = LinearRegressionModel(lags_future_covariates=[0])

# Encapsulation dans l'adapter
estimator = DartsAdapter(model=darts_model)

# Entrainement du modèle (syntaxe sklearn)
estimator.fit(X_train, y_train)

# Prédiction du modèle (syntaxe sklearn)
y_pred_darts = estimator.predict(X_test)

y_pred_darts

L'adapter `DartsAdapter` s'intègre naturellement dans les `Pipeline` de `sklearn.pipeline`, permettant de combiner des transformations sklearn avec des modèles darts.

In [ ]:
# Importation de la pipeline
from sklearn.pipeline import Pipeline
# Importation du transformer
from sklearn.preprocessing import StandardScaler
# Importation de l'adapter et du modèle darts
from tsforecast.adapters import DartsAdapter
from darts.models import XGBModel 

# Construction de la pipeline avec preprocessing sklearn et modèle darts
pipeline_darts = Pipeline([
    ('scaler', StandardScaler()),  # Preprocessing sklearn
    ('forecaster', DartsAdapter(model=XGBModel(lags_future_covariates=[0])))  # Modèle darts encapsulé
])

# Propagation du format pandas à tous les transformers de la pipeline
pipeline_darts.set_output(transform="pandas")

# Entraînement de la pipeline
pipeline_darts.fit(X_train, y_train)

# Prédiction
y_pred_pipeline_darts = pipeline_darts.predict(X_test)

y_pred_pipeline_darts

`GridSearchCV` de sklearn peut être utilisé pour optimiser les hyperparamètres des modèles darts encapsulés dans l'adapter.

In [ ]:
# Importation de GridSearchCV
from sklearn.model_selection import GridSearchCV
# Importation de l'adapter et du modèle
from tsforecast.adapters import DartsAdapter
from darts.models import XGBModel

# Définition de la pipeline avec modèle darts
pipeline_darts_grid = Pipeline([
    ('scaler', StandardScaler()),
    ('forecaster', DartsAdapter(model=XGBModel(lags_future_covariates=[0])))
])

# Propagation du format pandas à tous les transformers de la pipeline
pipeline_darts_grid.set_output(transform="pandas")

# Définition de la grille de paramètres
# Note: Utiliser le préfixe 'forecaster__model__' pour accéder aux paramètres du modèle darts
# Les paramètres forcés par l'adapter (lags, lags_past_covariates,
# lags_future_covariates, output_chunk_length) ne sont pas explorés ici.
# On se concentre sur les hyperparamètres propres au modèle de boosting.
param_grid = {
    'forecaster__model__n_estimators': [50, 100, 200],
    'forecaster__model__max_depth': [3, 5, 7],
    'forecaster__model__learning_rate': [0.01, 0.05, 0.1],
}

# Initialisation du GridSearchCV
grid_search_darts = GridSearchCV(
    estimator=pipeline_darts_grid,
    param_grid=param_grid,
    cv=cv_val,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)

# Entraînement
grid_search_darts.fit(X_train, y_train)

# Meilleurs paramètres et score
print(f"Meilleurs paramètres: {grid_search_darts.best_params_}")
print(f"Meilleur score: {grid_search_darts.best_score_:.4f}")

# Prédiction avec le meilleur modèle
y_pred_grid_darts = grid_search_darts.predict(X_test)

y_pred_grid_darts

### 3.6. Intégration des modèles de `hierarchicalforecast`


Le package `hierarchicalforecast` est spécialisé dans la réconciliation de prévisions hiérarchiques. 
Lorsque des séries temporelles sont liées par des contraintes d'agrégation (géographiques, temporelles, 
catégorielles), les prévisions indépendantes de chaque série ne respectent pas nécessairement ces contraintes. 
La réconciliation ajuste les prévisions pour les rendre **cohérentes**.

Pour intégrer ces fonctionnalités dans un workflow unifié, `tsforecast` fournit l'adapter 
`HierarchicalForecastAdapter` qui encapsule les méthodes de réconciliation dans une interface compatible `sklearn`.

La syntaxe devient alors celle de `sklearn` :
- `.fit(X, y)` pour construire la hiérarchie à partir des données historiques ;
- `.predict(X)` pour réconcilier des prévisions de base.

**Workflow typique :**
1. Entraîner des modèles indépendants pour chaque série de la hiérarchie
2. Générer des prévisions de base (potentiellement incohérentes)
3. Réconcilier ces prévisions avec l'adapter

Deux types de hiérarchies sont supportés :
- **Cross-sectionnelle** : agrégation entre séries (ex: Total = Composante_B + Composante_C)
- **Temporelle** : agrégation entre fréquences (ex: trimestriel = somme des mensuels)


#### 3.6.1. Réconciliation cross-sectionnelle

Dans cet exemple, on dispose de séries `value_b` et `value_c` dont la somme définit `value_a`. 
On entraîne des modèles indépendants sur chaque série, on génère des prévisions out-of-sample 
via `cross_val_predict`, puis on réconcilie ces prévisions pour restaurer la cohérence hiérarchique.


#### 3.6.1.1 Génération des prévisions indépendantes

In [ ]:
# Étape 1 : Préparation des features pour chaque composante
# Construction de covariables simples (trend, mois, lags) pour chaque série

# Fonction de construction des features pour la prédiction de chaque valeur
def build_features(series, horizon):
    """Build lag-based features for a univariate time series.

    Args:
        series: Time series with DatetimeIndex.
        horizon: Forecast horizon (number of steps ahead).

    Returns:
        Tuple of (X, y) aligned for the given horizon.
    """
    # Initialisation du DataFrame de features
    df_feat = pd.DataFrame(index=series.index)
    # Création des features
    df_feat['trend'] = np.arange(len(series))
    df_feat['month'] = series.index.get_level_values(-1).month
    for lag in [1, 2, 3, 6, 12]:
        df_feat[f'lag_{lag}'] = series.shift(lag)
    # Décalage pour aligner X[t] avec y[t+horizon]
    y_aligned = series.shift(-horizon)
    # Suppression des NaN
    mask = df_feat.notna().all(axis=1) & y_aligned.notna()
    return df_feat.loc[mask], y_aligned.loc[mask]

# Horizon de prédiction
horizon = 2
# Création des features pour chaque composante (niveau bottom) et le total
X_b, y_b = build_features(series=df_hier_wide['value_b'], horizon=horizon)
X_c, y_c = build_features(series=df_hier_wide['value_c'], horizon=horizon)
# Covariables pour le total (entraînement d'un modèle indépendant sur le total)
X_a, y_a = build_features(series=df_hier_wide['value_a'], horizon=horizon)

# Application de l'horizon de prédiction
X_a = X_a.shift(-horizon)
X_b = X_b.shift(-horizon)
X_c = X_c.shift(-horizon)

# Affichage
print(f'Covariables construites : X_b {X_b.shape}, X_c {X_c.shape}, X_a {X_a.shape}')
print(f'Target construites : y_b {y_b.shape}, y_c {y_c.shape}, y_a {y_a.shape}')

In [ ]:
# Étape 2 : Génération de prévisions out-of-sample via cross_val_predict
# Importation du modèle de prévision
from sklearn.linear_model import LinearRegression
from sklearn.utils import _safe_indexing

# Définition des dates de test
test_dates = [d.strftime('%Y-%m-%d') for d in pd.date_range('2022-01-01', periods=24, freq='MS')]

# Initialisation du modèle et de la crossval
model = LinearRegression()
cv_hier = PanelOutOfSampleSplit(
    test_indices=test_dates,
    test_size=1,
    gap=horizon,
)

# Collecte des prédictions par split
records_hat = []

# Boucle manuelle sur les splits
for train_idx, test_idx in cv_hier.split(X_a, y_a):
    # Extraction des jeux d'entraînement et de test
    X_a_train, y_a_train = _safe_indexing(X_a, train_idx), _safe_indexing(y_a, train_idx)
    X_b_train, y_b_train = _safe_indexing(X_b, train_idx), _safe_indexing(y_b, train_idx)
    X_c_train, y_c_train = _safe_indexing(X_c, train_idx), _safe_indexing(y_c, train_idx)

    X_a_test = _safe_indexing(X_a, test_idx)
    X_b_test = _safe_indexing(X_b, test_idx)
    X_c_test = _safe_indexing(X_c, test_idx)

    # Entraînement et prédiction pour chaque composante
    y_hat_a = model.fit(X_a_train, y_a_train).predict(X_a_test)
    y_hat_b = model.fit(X_b_train, y_b_train).predict(X_b_test)
    y_hat_c = model.fit(X_c_train, y_c_train).predict(X_c_test)

    # Reconstruction de l'index de test (MultiIndex entity/date)
    test_index = X_a.iloc[test_idx].index

    for (entity, date), va, vb, vc in zip(test_index, y_hat_a, y_hat_b, y_hat_c):
        records_hat.append({
            'entity': entity, 'date': date,
            'y_hat_a': va, 'y_hat_b': vb, 'y_hat_c': vc,
        })

# DataFrame de prédictions avec MultiIndex (entity, date)
df_hat = pd.DataFrame(records_hat).set_index(['entity', 'date'])

# Vérification de l'incohérence des prévisions indépendantes
incoherence = (df_hat['y_hat_a'] - (df_hat['y_hat_b'] + df_hat['y_hat_c'])).abs()
print('Prévisions out-of-sample générées :')
print(f'  Shape : {df_hat.shape}')
print(f'  Incohérence moyenne |ŷ_a - (ŷ_b + ŷ_c)| = {incoherence.mean():.4f}')
print(f'  Incohérence max = {incoherence.max():.4f}')

#### 3.6.1.2 Réconciliation des prévisions

In [ ]:
# ========================================================================
# Étape 3 : Réconciliation des prévisions
# ========================================================================
# Construction du DataFrame de prévisions au format MultiIndex attendu par l'adapter.
# Le MultiIndex doit reprendre la même structure que df_hier_long['y'] :
#   niveaux (entity, category, component, date), du plus agrégé au plus granulaire.
# Pour les lignes de niveau agrégé (ici entity/category), le niveau "component"
# absent est représenté par une chaîne vide "". L'adapter construit le
# unique_id = "DE/Total" en filtrant les parties vides lors de la jointure par "/".

records_hat_hier = []

for (entity, date), row in df_hat.iterrows():
    # Prévision au niveau agrégé (entity/category) — component absent → ""
    records_hat_hier.append({
        'entity': entity, 'category': 'Total', 'component': '', 'date': date,
        'LinearRegression': row['y_hat_a'],
    })
    # Prévision au niveau bottom pour la composante B
    records_hat_hier.append({
        'entity': entity, 'category': 'Total', 'component': 'B', 'date': date,
        'LinearRegression': row['y_hat_b'],
    })
    # Prévision au niveau bottom pour la composante C
    records_hat_hier.append({
        'entity': entity, 'category': 'Total', 'component': 'C', 'date': date,
        'LinearRegression': row['y_hat_c'],
    })

# DataFrame de prévisions avec MultiIndex (entity, category, component, date)
Y_hat = (
    pd.DataFrame(records_hat_hier)
    .set_index(['entity', 'category', 'component', 'date'])
)

# Affichage
print(f'DataFrame de prévisions : {Y_hat.shape}')
print(Y_hat.head(6))

In [ ]:
df_hier_long['y'].head()

In [ ]:
# Importation de l'adapter et des méthodes de réconciliation
from tsforecast.adapters import HierarchicalForecastAdapter
from hierarchicalforecast.methods import BottomUp, MinTrace

# Initialisation de l'adapter avec la spécification cross-sectionnelle.
# La spécification inclut le niveau entité en préfixe de chaque niveau de la hiérarchie,
# du niveau le plus agrégé (entity/category) au niveau bottom (entity/category/component).
spec_cs = [['entity', 'category'], ['entity', 'category', 'component']]

adapter_cs = HierarchicalForecastAdapter(
    reconcilers=[BottomUp(), MinTrace(method='ols')],
    spec=spec_cs,
)

# Entraînement sur les données historiques bottom-level.
# L'adapter calcule automatiquement les agrégats intermédiaires via aggregate().
adapter_cs.fit(None, df_hier_long['y'])

# Informations sur la hiérarchie construite
info = adapter_cs.get_hierarchy_info()
print(f'Hiérarchie construite :')
print(f'  Nombre total de séries : {info["n_series"]}')
print(f'  Séries bottom-level : {info["n_bottom"]}')
print(f'  Niveaux : {info["levels"]}')
print(f'  Tags : {info["tags"]}')

# Réconciliation à partir du DataFrame au format MultiIndex
Y_reconciled = adapter_cs.predict(Y_hat)

# Affichage
print(f'\nPrévisions réconciliées : {Y_reconciled.shape}')
print(f'Colonnes : {Y_reconciled.columns.tolist()}')

# Vérification de la cohérence après réconciliation
# Choix de l'entité
entity = "DE"
# Parcours des colonnes de méthodes (colonne contenant "/" = méthode de réconciliation)
for method_col in [c for c in Y_reconciled.columns if '/' in c]:
    # Extraction des valeurs pour les trois niveaux de la hiérarchie
    total = Y_reconciled[Y_reconciled['unique_id'] == f'{entity}/Total'][method_col].values
    b_vals = Y_reconciled[Y_reconciled['unique_id'] == f'{entity}/Total/B'][method_col].values
    c_vals = Y_reconciled[Y_reconciled['unique_id'] == f'{entity}/Total/C'][method_col].values
    max_err = np.max(np.abs(total - (b_vals + c_vals)))
    # Affichage de l'erreur de cohérence résiduelle
    print(f'  {method_col}: max |Total - (B+C)| = {max_err:.1e}')

#### 3.6.2. Réconciliation temporelle

La réconciliation temporelle agrège les prévisions à différentes fréquences. 
Par exemple, des prévisions mensuelles sont agrégées en prévisions trimestrielles. 
La réconciliation ajuste les prévisions pour que la somme des mensuels corresponde aux trimestriels.

L'adapter détecte automatiquement la colonne d'identifiant lorsque le `id_col` par défaut 
(`'unique_id'`) ne correspond pas au nom du niveau du MultiIndex.


In [ ]:
# Utilisation des données de panel générées précédemment (section 1)
# y_train contient des données mensuelles avec MultiIndex (entity, date)

# Spécification de la hiérarchie temporelle
spec_temporal = {
    'quarterly': 3,  # Agrégation trimestrielle (somme de 3 mois)
    'monthly': 1,    # Niveau de base (mensuel)
}

# Initialisation de l'adapter
# /!\ id_col='entity' correspond au nom du niveau du MultiIndex
# Alternativement, l'adapter détecte automatiquement si 'unique_id' est absent
adapter_temporal = HierarchicalForecastAdapter(
    reconcilers=[BottomUp(), MinTrace(method='ols')],
    spec=spec_temporal,
    aggregation_type='local'
)

# Construction de la hiérarchie temporelle
adapter_temporal.fit(None, y_train)

# Informations sur la hiérarchie
info_temp = adapter_temporal.get_hierarchy_info()
print(f'Hiérarchie temporelle construite :')
print(f'  Nombre total de séries : {info_temp["n_series"]}')
print(f'  Séries bottom-level : {info_temp["n_bottom"]}')
print(f'  Niveaux : {info_temp["levels"]}')

# Aperçu des données agrégées
print(f'\nAperçu Y_df_ (données agrégées) :')
print(adapter_temporal.Y_df_.head(10))

#### 3.6.3. Évaluation avec `cross_val_score`

L'adapter est compatible avec `cross_val_score` de `sklearn`. Dans ce contexte :
- `X` contient les **prévisions de base** à réconcilier
- `y` contient les **valeurs réelles** pour évaluer la qualité
- `fit(X_train, y_train)` construit la hiérarchie à partir de `y_train`
- `score(X_test, y_test)` réconcilie `X_test` et compare à `y_test`

Pour cet exemple simplifié, on utilise les valeurs réelles comme prévisions de base 
(i.e. les "prévisions" sont déjà parfaites avant réconciliation).


In [ ]:
# Importation de cross_val_score
from sklearn.model_selection import cross_val_score

# Définition de l'adapter avec hiérarchie temporelle
adapter_cv = HierarchicalForecastAdapter(
    reconcilers=[BottomUp()],
    spec={'quarterly': 3, 'monthly': 1},
)

# Évaluation avec cross-validation
# X=y : les "prévisions de base" sont les vraies valeurs (cas simplifié)
scores_hierarchical = cross_val_score(
    estimator=adapter_cv,
    X=y,
    y=y,
    cv=cv,
    scoring='r2'
)

print(f'Scores R² par fold : {scores_hierarchical}')
print(f'Score R² moyen : {scores_hierarchical.mean():.4f} (+/- {scores_hierarchical.std():.4f})')


#### 3.6.4. Intégration dans une `Pipeline` sklearn

L'adapter peut être intégré comme étape finale d'une `Pipeline` sklearn. 
Les étapes de preprocessing transforment les prévisions de base avant réconciliation.

**Attention :** Dans cette configuration, `X` représente les prévisions de base (et non des covariables) 
et les transformers s'appliquent à ces prévisions. Le `fit` de l'adapter utilise `y` pour construire la hiérarchie.


In [ ]:
# Importation de la pipeline et du scaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Construction d'une pipeline : preprocessing + réconciliation
pipeline_hier = Pipeline([
    ('scaler', StandardScaler()),
    ('reconciler', HierarchicalForecastAdapter(
        reconcilers=[BottomUp()],
        spec={'quarterly': 3, 'monthly': 1},
    ))
])

# Propagation du format pandas
pipeline_hier.set_output(transform='pandas')

# Entraînement de la pipeline
# /!\ X=y car les prévisions de base sont ici les valeurs réelles
pipeline_hier.fit(X=y_train, y=y_train)

# Prédiction (réconciliation des prévisions transformées)
y_pred_pipeline_hier = pipeline_hier.predict(X_test)

# Affichage
print(f'Prédictions via Pipeline : {type(y_pred_pipeline_hier)}')
if hasattr(y_pred_pipeline_hier, 'shape'):
    print(f'Shape : {y_pred_pipeline_hier.shape}')
if hasattr(y_pred_pipeline_hier, 'head'):
    print(y_pred_pipeline_hier.head())

#### 3.6.5. Optimisation avec `GridSearchCV`

`GridSearchCV` peut être utilisé pour comparer différentes configurations de réconciliation :
- Type d'agrégation (`local` vs `global`)
- Spécification temporelle (différents niveaux d'agrégation)

**Note :** Les méthodes de réconciliation (`reconcilers`) étant des objets instanciés, 
leur exploration nécessite de passer des listes complètes comme valeurs de la grille.


In [ ]:
# Importation de GridSearchCV
from sklearn.model_selection import GridSearchCV

# Définition de la grille de paramètres
# Comparaison de différentes méthodes de réconciliation et types d'agrégation
param_grid = {
    'reconcilers': [
        [BottomUp()],
        [MinTrace(method='ols')],
        [MinTrace(method='mint_shrink')],
    ],
    'aggregation_type': ['local'],
}

# Initialisation de l'estimateur de base
adapter_grid = HierarchicalForecastAdapter(
    reconcilers=[BottomUp()],
    spec={'quarterly': 3, 'monthly': 1},
    id_col='entity',
)

# Initialisation du GridSearchCV
grid_search_hier = GridSearchCV(
    estimator=adapter_grid,
    param_grid=param_grid,
    cv=cv_val,
    n_jobs=1,  # Sérialisation pour éviter les conflits de mémoire
    scoring='r2',
    error_score='raise',
)

# Entraînement avec recherche d'hyperparamètres
# X=y car les prévisions de base sont ici les valeurs réelles
grid_search_hier.fit(X=y, y=y)

# Résultats
print(f'Meilleurs paramètres : {grid_search_hier.best_params_}')
print(f'Meilleur score R² : {grid_search_hier.best_score_:.4f}')

# Détails des résultats
results = pd.DataFrame(grid_search_hier.cv_results_)
print(f'\nRésumé des résultats :')
print(results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']])


### 3.7. Intégration des modèles de `opera`

Le package `opera` se spécialise dans l'agrégation en ligne d'experts (online expert aggregation), permettant de combiner dynamiquement les prévisions de plusieurs modèles. Son API spécifique nécessite des formats particuliers.

Pour intégrer ces fonctionnalités dans un workflow unifié, `tsforecast` fournit l'adapter `OperaAdapter` qui encapsule les algorithmes d'agrégation dans une interface compatible `sklearn`.

La syntaxe devient alors celle de `sklearn` :
- `.fit(X, y)` pour entraîner le système d'agrégation sur les prédictions des experts ;
- `.predict(X)` pour combiner les prédictions des experts ;
- `.partial_fit(X, y)` pour mise à jour incrémentale (online learning) ;
- `.score(X, y)` pour évaluer la qualité de l'ensemble ;

L'adapter gère automatiquement :
- La conversion des formats de données
- L'initialisation des coefficients d'experts
- La mise à jour des poids selon les performances passées
- Le calcul des prédictions d'ensemble

**Cas d'usage typique:** Vous avez entraîné plusieurs modèles différents (ex: Ridge, Lasso, RandomForest) et vous voulez les combiner de manière optimale en fonction de leurs performances.

In [ ]:
# Importation de l'adapter
from tsforecast.adapters import OperaAdapter
# Importation de modèles sklearn pour créer des "experts"
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

# Étape 1: Entraîner plusieurs modèles "experts"
print("Entraînement des modèles experts...")
expert_ridge = Ridge(alpha=1.0).fit(X_train, y_train)
expert_lasso = Lasso(alpha=1.0).fit(X_train, y_train)
expert_rf = RandomForestRegressor(n_estimators=50, random_state=42).fit(X_train, y_train)

# Étape 2: Obtenir leurs prédictions (qui serviront d'input à opera)
expert_predictions_train = np.column_stack([
    expert_ridge.predict(X_train),
    expert_lasso.predict(X_train),
    expert_rf.predict(X_train)
])

expert_predictions_test = np.column_stack([
    expert_ridge.predict(X_test),
    expert_lasso.predict(X_test),
    expert_rf.predict(X_test)
])

print(f"Prédictions des experts (train): {expert_predictions_train.shape}")
print(f"Prédictions des experts (test): {expert_predictions_test.shape}")

# Étape 3: Entraîner l'adapter Opera pour combiner les experts
estimator_opera = OperaAdapter(
    model="BOA",  # Bernstein Online Aggregation
    loss_type="mse"
)

# Entraînement sur les prédictions des experts
estimator_opera.fit(expert_predictions_train, y_train.values)

# Étape 4: Combiner les prédictions des experts sur le test
y_pred_opera = estimator_opera.predict(expert_predictions_test)

# Évaluation
r2_opera = estimator_opera.score(expert_predictions_test, _safe_indexing(y, test).values)
print(f"\nScore R² de l'ensemble: {r2_opera:.4f}")

# Comparaison avec les experts individuels
from sklearn.metrics import r2_score
print(f"Score R² Ridge: {r2_score(_safe_indexing(y, test), expert_ridge.predict(X_test)):.4f}")
print(f"Score R² Lasso: {r2_score(_safe_indexing(y, test), expert_lasso.predict(X_test)):.4f}")
print(f"Score R² RandomForest: {r2_score(_safe_indexing(y, test), expert_rf.predict(X_test)):.4f}")

y_pred_opera

`GridSearchCV` peut être utilisé pour optimiser le choix de l'algorithme d'agrégation et du type de perte.

In [ ]:
# Importation de GridSearchCV
from sklearn.model_selection import GridSearchCV
from tsforecast.adapters import OperaAdapter

# Préparation des prédictions des experts sur l'ensemble complet
expert_predictions_full = np.column_stack([
    expert_ridge.predict(X),
    expert_lasso.predict(X),
    expert_rf.predict(X)
])

# Définition de la grille de paramètres
param_grid = {
    'model': ['BOA', 'EWA', 'MLpol'],  # Différents algorithmes d'agrégation
    'loss_type': ['mse', 'mae'],  # Différentes fonctions de perte
}

# Initialisation du GridSearchCV
grid_search_opera = GridSearchCV(
    estimator=OperaAdapter(),
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)

# Entraînement
print("Recherche des meilleurs hyperparamètres...")
grid_search_opera.fit(expert_predictions_full, y.values)

# Meilleurs paramètres et score
print(f"\nMeilleurs paramètres: {grid_search_opera.best_params_}")
print(f"Meilleur score: {grid_search_opera.best_score_:.4f}")

# Prédiction avec le meilleur modèle
y_pred_grid_opera = grid_search_opera.predict(expert_predictions_test)

# Comparaison des performances
r2_grid = r2_score(_safe_indexing(y, test), y_pred_grid_opera)
print(f"Score R² (meilleur modèle): {r2_grid:.4f}")

y_pred_grid_opera

Une particularité d'`OperaAdapter` est le support de l'apprentissage incrémental via `partial_fit`, permettant de mettre à jour les poids des experts au fur et à mesure que de nouvelles données arrivent (online learning).

In [ ]:
# Simulation d'un scénario d'apprentissage incrémental
# Divisons les données d'entraînement en batches

# Initialisation de l'adapter
estimator_opera_online = OperaAdapter(model="EWA", loss_type="mse")  # Exponentially Weighted Average

# Premier batch: entraînement initial
batch_size = len(X_train) // 3
expert_preds_batch1 = expert_predictions_train[:batch_size]
y_batch1 = y_train.values[:batch_size]

print(f"Entraînement initial sur {batch_size} observations...")
estimator_opera_online.fit(expert_preds_batch1, y_batch1)

# Batches suivants: mise à jour incrémentale
for i in range(1, 3):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if i < 2 else len(X_train)
    
    expert_preds_batch = expert_predictions_train[start_idx:end_idx]
    y_batch = y_train.values[start_idx:end_idx]
    
    print(f"Mise à jour incrémentale {i}: {len(y_batch)} observations...")
    estimator_opera_online.partial_fit(expert_preds_batch, y_batch)

# Prédiction finale
y_pred_opera_online = estimator_opera_online.predict(expert_predictions_test)

# Évaluation
r2_online = estimator_opera_online.score(expert_predictions_test, _safe_indexing(y, test).values)
print(f"\nScore R² (apprentissage incrémental): {r2_online:.4f}")

y_pred_opera_online